# 07 — GAM Lag 0–7

Baseline spline-Poisson model cho Lag0–Lag7; kiểm tra overdispersion trước sensitivity analysis.

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
import statsmodels.api as sm
import patsy
PROJECT_ROOT=Path.cwd().parents[2]; SON_OUTPUT=PROJECT_ROOT/'notebooks'/'son_nb'/'output'; WORK_DATA=SON_OUTPUT/'data'; WORK_TABLE=SON_OUTPUT/'tables'; WORK_FIG=SON_OUTPUT/'figures'
health=pd.read_csv(WORK_DATA/'beijing_daily_health_simulated_working.csv',parse_dates=['datetime']).sort_values('datetime')
for lag in range(8): health[f'pm25_lag{lag}']=health['pm2.5'].shift(lag)

In [ ]:
def fit_lag(data,lag):
    d=data.dropna(subset=['resp_cases',f'pm25_lag{lag}','TEMP','DEWP','time_index','dayofweek'])
    formula=(f"resp_cases ~ bs(time_index, df=8, degree=3) + bs(TEMP, df=6, degree=3) + bs(DEWP, df=6, degree=3) + C(dayofweek) + pm25_lag{lag}")
    y,X=patsy.dmatrices(formula,data=d,return_type='dataframe')
    model=sm.GLM(y,X,family=sm.families.Poisson()).fit()
    term=f'pm25_lag{lag}'; b=model.params[term]; se=model.bse[term]; lo=b-1.96*se; hi=b+1.96*se
    return {'lag':lag,'n':len(d),'beta_per_ug':b,'RR_per_10ug':np.exp(b*10),'CI_low':np.exp(lo*10),'CI_high':np.exp(hi*10),'p_value':model.pvalues[term],'pearson_dispersion':np.sum(model.resid_pearson**2)/model.df_resid,'AIC':model.aic}
res=pd.DataFrame([fit_lag(health,l) for l in range(8)]); display(res); res.to_csv(WORK_TABLE/'beijing_gam_lag_results_working.csv',index=False)

In [ ]:
fig,ax=plt.subplots(figsize=(10,5)); y=res.RR_per_10ug; ax.errorbar(res.lag,y,yerr=[y-res.CI_low,res.CI_high-y],fmt='o-',capsize=4); ax.axhline(1,ls='--'); ax.set_xlabel('Lag (days)'); ax.set_ylabel('RR per +10 µg/m³ PM2.5'); ax.set_title('Beijing — PM2.5 Lag Effect (Synthetic Health)'); fig.tight_layout(); fig.savefig(WORK_FIG/'beijing_gam_lag_effect_working.png',dpi=200); plt.show()

### Lưu ý
Đây là Poisson spline-GAM-style baseline. Nếu dispersion lớn, bước sau sẽ đánh giá quasi-Poisson/Negative Binomial.